# Prefix Sum

Prefix sum is a powerful technique for efficiently answering range queries and finding subarrays with specific properties. Combined with hash maps, it becomes one of the most versatile tools for solving subarray problems.

---

## The Core Theory

### What is Prefix Sum?

A **prefix sum array** (also called cumulative sum) stores the running total where each element `prefix[i]` represents the sum of all elements from index 0 to i.

```
Index:     0   1   2   3   4
Original: [1,  2,  3,  4,  5]
Prefix:   [1,  3,  6, 10, 15]
           ↑   ↑   ↑   ↑   ↑
           1  1+2 1+2+3 ...
```

### The Mathematical Foundation

**Definition**: For an array `A[0...n-1]`, the prefix sum array `P` is defined as:
- `P[0] = A[0]`
- `P[i] = P[i-1] + A[i]` for i > 0

Or equivalently: `P[i] = A[0] + A[1] + ... + A[i] = Σ A[k]` for k from 0 to i

### The Key Property: Range Sum in O(1)

The most important property of prefix sums is:

```
Sum of A[i...j] = P[j] - P[i-1]
```

**Why does this work?**
```
P[j]   = A[0] + A[1] + ... + A[i-1] + A[i] + ... + A[j]
P[i-1] = A[0] + A[1] + ... + A[i-1]
─────────────────────────────────────────────────────────
P[j] - P[i-1] = A[i] + A[i+1] + ... + A[j]  ✓
```

### Why Use prefix[0] = 0?

In practice, we often prepend a 0 to the prefix array:

```
Index:     0   1   2   3   4   5
Original:     [1,  2,  3,  4,  5]
Prefix:   [0,  1,  3,  6, 10, 15]
```

This simplifies the formula to: `Sum of A[i...j] = P[j+1] - P[i]`

And handles edge cases like "sum from index 0" without special logic.

In [ ]:
def build_prefix_sum(nums):
    """Build prefix sum array: O(n)"""
    prefix = [0] * (len(nums) + 1)  # Extra element for easier calculation
    for i in range(len(nums)):
        prefix[i + 1] = prefix[i] + nums[i]
    return prefix

def range_sum(prefix, left, right):
    """Get sum of nums[left:right+1] in O(1)"""
    return prefix[right + 1] - prefix[left]

# Example
nums = [1, 2, 3, 4, 5]
prefix = build_prefix_sum(nums)
print(f"Original: {nums}")
print(f"Prefix:   {prefix}")
print(f"Sum [1,3]: {range_sum(prefix, 1, 3)}")  # 2+3+4 = 9

---

## Why Combine Prefix Sum with Hash Map?

### The Problem with Naive Approach

To find all subarrays with sum = k, the naive approach checks every possible subarray:

```python
# O(n²) or O(n³) - Too slow!
for i in range(n):
    for j in range(i, n):
        if sum(nums[i:j+1]) == k:
            count += 1
```

### The Hash Map Insight

**Key observation**: If we know `prefix[j] = S` and we're looking for subarrays with sum = k, we need to find all previous indices `i` where `prefix[i] = S - k`.

```
prefix[j] - prefix[i] = k
       ↓
prefix[i] = prefix[j] - k
```

**Hash map stores**: `{prefix_sum: count}` or `{prefix_sum: first_index}`

This transforms O(n²) → O(n) because hash map lookup is O(1)!

### Visual Example

Array: `[1, 2, 3, -3, 4]`, find subarrays with sum = 3

```
Index:      0    1    2    3    4
Array:     [1,   2,   3,  -3,   4]
Prefix:    [1,   3,   6,   3,   7]

At index 2: prefix = 6, looking for 6-3 = 3
  → Found at index 1! Subarray [2:2] = [3] ✓

At index 3: prefix = 3, looking for 3-3 = 0
  → Found at index -1 (initial)! Subarray [0:3] = [1,2,3,-3] ✓
  
At index 4: prefix = 7, looking for 7-3 = 4
  → Not found
```

---

## Hash Map Strategies: Count vs First Index

### Strategy 1: Count Occurrences (for counting subarrays)

```python
prefix_count = {0: 1}  # Initialize with sum 0 occurring once
```

**Use when**: You need to COUNT how many subarrays have sum = k

**Why count?** Multiple subarrays can end at the same index with the same sum.

### Strategy 2: First Index (for longest subarray)

```python
prefix_map = {0: -1}  # Initialize with sum 0 at index -1
```

**Use when**: You need the LONGEST subarray with sum = k

**Why first index?** To maximize `j - i`, we want the smallest `i`, which is the first occurrence.

### Comparison

| Goal | Hash Map Value | Initialize | Update Rule |
|------|---------------|------------|-------------|
| Count subarrays | Frequency | `{0: 1}` | Always increment |
| Longest subarray | First index | `{0: -1}` | Only store if not exists |

---

## Pattern 1: Range Sum Query

Answer multiple range sum queries in O(1) each after O(n) preprocessing.

In [ ]:
class NumArray:
    """LeetCode 303: Range Sum Query - Immutable"""
    
    def __init__(self, nums):
        self.prefix = [0]
        for num in nums:
            self.prefix.append(self.prefix[-1] + num)
    
    def sum_range(self, left, right):
        return self.prefix[right + 1] - self.prefix[left]

# Test
arr = NumArray([1, 2, 3, 4, 5])
print(f"Sum [0,2]: {arr.sum_range(0, 2)}")  # 1+2+3 = 6
print(f"Sum [2,4]: {arr.sum_range(2, 4)}")  # 3+4+5 = 12

---

## Pattern 2: Subarray Sum Equals K

Find subarrays with a specific sum using **prefix sum + hash map**.

### The Algorithm Step by Step

1. Initialize hash map with `{0: 1}` (empty prefix has sum 0)
2. Maintain running prefix sum
3. At each index, check if `prefix_sum - k` exists in hash map
4. If yes, add its count to result (those are valid starting points)
5. Add current prefix_sum to hash map

### Why Initialize with {0: 1}?

This handles subarrays that start from index 0.

```
Array: [3, 4, 7], k = 3
At index 0: prefix = 3, looking for 3-3 = 0
  → Without {0: 1}, we'd miss the subarray [3]!
```

### Detailed Walkthrough

```
Array: [1, 1, 1], k = 2

Step 0: prefix_count = {0: 1}, count = 0

Step 1: num = 1
  prefix_sum = 1
  Looking for 1 - 2 = -1 → Not in map → count += 0
  prefix_count = {0: 1, 1: 1}

Step 2: num = 1
  prefix_sum = 2
  Looking for 2 - 2 = 0 → In map with count 1 → count += 1
  prefix_count = {0: 1, 1: 1, 2: 1}

Step 3: num = 1
  prefix_sum = 3
  Looking for 3 - 2 = 1 → In map with count 1 → count += 1
  prefix_count = {0: 1, 1: 1, 2: 1, 3: 1}

Result: 2 subarrays ([1,1] at indices 0-1 and [1,1] at indices 1-2)
```

In [ ]:
from collections import defaultdict

def subarray_sum(nums, k):
    """LeetCode 560: Count subarrays with sum = k"""
    count = 0
    prefix_sum = 0
    prefix_count = defaultdict(int)
    prefix_count[0] = 1  # Empty prefix has sum 0
    
    for num in nums:
        prefix_sum += num
        # If prefix_sum - k exists, we found subarrays with sum k
        count += prefix_count[prefix_sum - k]
        prefix_count[prefix_sum] += 1
    
    return count

print(f"[1,1,1] k=2: {subarray_sum([1,1,1], 2)}")  # 2
print(f"[1,2,3] k=3: {subarray_sum([1,2,3], 3)}")  # 2 ([1,2] and [3])

---

## Pattern 3: Longest Subarray with Sum = 0 (or K)

Find the **longest** subarray with a specific sum.

### Key Difference from Counting

| Counting | Longest |
|----------|---------|
| Store frequency of each prefix sum | Store first index of each prefix sum |
| `prefix_count[sum] += 1` | `if sum not in map: map[sum] = i` |
| Add all occurrences | Only keep first occurrence |

### Why First Occurrence?

To maximize length `j - i`, we want the smallest possible `i`.

```
Array: [1, -1, 1, -1, 1, -1]
Prefix: [1, 0, 1, 0, 1, 0]

Sum 0 appears at indices: 1, 3, 5
Sum 1 appears at indices: 0, 2, 4

At index 5 (prefix = 0):
  - If we stored first occurrence (index 1): length = 5 - 1 = 4
  - If we stored last occurrence (index 3): length = 5 - 3 = 2 ✗
```

### Why Initialize with {0: -1}?

This handles subarrays that start from index 0.

```
Array: [1, -1], looking for sum = 0
Prefix at index 1 = 0

With {0: -1}: length = 1 - (-1) = 2 ✓ (entire array)
Without: We'd miss it!
```

### Generalization: Longest Subarray with Sum = K

```python
def longest_subarray_sum_k(nums, k):
    prefix_map = {0: -1}
    prefix_sum = 0
    max_len = 0
    
    for i, num in enumerate(nums):
        prefix_sum += num
        
        # Looking for prefix_sum - k
        if (prefix_sum - k) in prefix_map:
            max_len = max(max_len, i - prefix_map[prefix_sum - k])
        
        if prefix_sum not in prefix_map:
            prefix_map[prefix_sum] = i
    
    return max_len
```

In [ ]:
def longest_subarray_sum_zero(nums):
    """Find longest subarray with sum = 0"""
    prefix_map = {0: -1}  # sum 0 at index -1
    prefix_sum = 0
    max_len = 0
    
    for i, num in enumerate(nums):
        prefix_sum += num
        
        if prefix_sum in prefix_map:
            max_len = max(max_len, i - prefix_map[prefix_sum])
        else:
            prefix_map[prefix_sum] = i  # Store first occurrence only
    
    return max_len

print(f"[1,-1,3,-3,2]: {longest_subarray_sum_zero([1,-1,3,-3,2])}")  # 4

---

## Pattern 4: Equal 0s and 1s (Contiguous Array)

Transform the problem: Replace 0 → -1, then find longest subarray with sum = 0.

### The Transformation Trick

**Original problem**: Find longest subarray with equal number of 0s and 1s.

**Insight**: If we replace every 0 with -1:
- Each 1 contributes +1
- Each 0 contributes -1
- Equal 0s and 1s means sum = 0!

```
Original: [0, 1, 0, 1, 1, 0]
Transform: [-1, 1, -1, 1, 1, -1]

Subarray [0,1,0,1] → [-1,1,-1,1] → sum = 0 ✓
  (2 zeros, 2 ones)
```

### Why This Works Mathematically

Let `c0` = count of 0s, `c1` = count of 1s in a subarray.

After transformation:
- Sum = `c1 * 1 + c0 * (-1)` = `c1 - c0`

For equal counts: `c0 = c1` → Sum = `c1 - c0 = 0` ✓

### Common Transformations

| Original Problem | Transformation | New Problem |
|-----------------|----------------|-------------|
| Equal 0s and 1s | 0 → -1 | Sum = 0 |
| More 1s than 0s | 0 → -1 | Sum > 0 |
| K more 1s than 0s | 0 → -1 | Sum = K |
| Equal vowels and consonants | vowel → 1, consonant → -1 | Sum = 0 |

In [ ]:
def find_max_length(nums):
    """LeetCode 525: Contiguous Array"""
    prefix_map = {0: -1}
    prefix_sum = 0
    max_len = 0
    
    for i, num in enumerate(nums):
        # Treat 0 as -1
        prefix_sum += 1 if num == 1 else -1
        
        if prefix_sum in prefix_map:
            max_len = max(max_len, i - prefix_map[prefix_sum])
        else:
            prefix_map[prefix_sum] = i
    
    return max_len

print(f"[0,1,0]: {find_max_length([0,1,0])}")  # 2
print(f"[0,1,1,0,1,1,1,0]: {find_max_length([0,1,1,0,1,1,1,0])}")  # 4

---

## Pattern 5: Binary Subarrays With Sum

Count subarrays in a binary array with a specific sum.

In [ ]:
def num_subarrays_with_sum(nums, goal):
    """LeetCode 930: Binary Subarrays With Sum"""
    count = 0
    prefix_sum = 0
    prefix_count = defaultdict(int)
    prefix_count[0] = 1
    
    for num in nums:
        prefix_sum += num
        count += prefix_count[prefix_sum - goal]
        prefix_count[prefix_sum] += 1
    
    return count

print(f"[1,0,1,0,1] goal=2: {num_subarrays_with_sum([1,0,1,0,1], 2)}")  # 4

---

## Pattern 6: 2D Prefix Sum

Extend prefix sum to 2D matrices for range sum queries.

In [ ]:
class NumMatrix:
    """LeetCode 304: Range Sum Query 2D - Immutable"""
    
    def __init__(self, matrix):
        if not matrix:
            return
        m, n = len(matrix), len(matrix[0])
        self.prefix = [[0] * (n + 1) for _ in range(m + 1)]
        
        for i in range(m):
            for j in range(n):
                self.prefix[i+1][j+1] = (matrix[i][j] 
                                         + self.prefix[i][j+1] 
                                         + self.prefix[i+1][j] 
                                         - self.prefix[i][j])
    
    def sum_region(self, row1, col1, row2, col2):
        return (self.prefix[row2+1][col2+1] 
                - self.prefix[row1][col2+1] 
                - self.prefix[row2+1][col1] 
                + self.prefix[row1][col1])

# Test
matrix = [
    [3, 0, 1, 4, 2],
    [5, 6, 3, 2, 1],
    [1, 2, 0, 1, 5],
    [4, 1, 0, 1, 7],
    [1, 0, 3, 0, 5]
]
nm = NumMatrix(matrix)
print(f"Sum region [2,1] to [4,3]: {nm.sum_region(2, 1, 4, 3)}")  # 8

---

## Pattern 7: Product Except Self

Use prefix and suffix products instead of sums.

In [ ]:
def product_except_self(nums):
    """LeetCode 238: Product of Array Except Self"""
    n = len(nums)
    result = [1] * n
    
    # Prefix products
    prefix = 1
    for i in range(n):
        result[i] = prefix
        prefix *= nums[i]
    
    # Suffix products
    suffix = 1
    for i in range(n - 1, -1, -1):
        result[i] *= suffix
        suffix *= nums[i]
    
    return result

print(f"[1,2,3,4]: {product_except_self([1,2,3,4])}")  # [24,12,8,6]

---

## Deep Dive: Hash Map Mechanics

### Time Complexity Analysis

| Operation | Without Hash Map | With Hash Map |
|-----------|-----------------|---------------|
| Check if prefix sum exists | O(n) scan | O(1) lookup |
| Find all matching indices | O(n) | O(1) |
| Overall algorithm | O(n²) | O(n) |

### Space Complexity

The hash map stores at most `n+1` unique prefix sums, so space is O(n).

**Worst case**: All prefix sums are unique (e.g., `[1, 2, 3, 4, 5]`)
**Best case**: Many repeated prefix sums (e.g., `[1, -1, 1, -1]`)

### Common Pitfalls

**1. Forgetting to initialize the hash map**
```python
# Wrong - misses subarrays starting at index 0
prefix_map = {}

# Correct
prefix_map = {0: -1}  # for longest
prefix_count = {0: 1}  # for counting
```

**2. Updating hash map before checking**
```python
# Wrong order
prefix_count[prefix_sum] += 1  # Update first
count += prefix_count[prefix_sum - k]  # Then check - WRONG!

# Correct order
count += prefix_count[prefix_sum - k]  # Check first
prefix_count[prefix_sum] += 1  # Then update
```

**3. Overwriting first occurrence when finding longest**
```python
# Wrong - always updates
prefix_map[prefix_sum] = i

# Correct - only store first occurrence
if prefix_sum not in prefix_map:
    prefix_map[prefix_sum] = i
```

---

## Summary: When to Use Prefix Sum

| Problem Type | Technique |
|-------------|----------|
| Multiple range sum queries | Build prefix array, query in O(1) |
| Count subarrays with sum = k | Prefix sum + hash map (count occurrences) |
| Longest subarray with sum = k | Prefix sum + hash map (first occurrence) |
| Equal 0s and 1s | Replace 0 → -1, find sum = 0 |
| 2D range queries | 2D prefix sum with inclusion-exclusion |
| Product queries | Prefix and suffix products |

---

## Related LeetCode Problems

| Problem | Difficulty | Pattern |
|---------|------------|--------|
| [303. Range Sum Query](https://leetcode.com/problems/range-sum-query-immutable/) | Easy | Basic prefix sum |
| [304. Range Sum Query 2D](https://leetcode.com/problems/range-sum-query-2d-immutable/) | Medium | 2D prefix sum |
| [238. Product of Array Except Self](https://leetcode.com/problems/product-of-array-except-self/) | Medium | Prefix/suffix product |
| [525. Contiguous Array](https://leetcode.com/problems/contiguous-array/) | Medium | Transform + longest subarray |
| [560. Subarray Sum Equals K](https://leetcode.com/problems/subarray-sum-equals-k/) | Medium | Count subarrays |
| [930. Binary Subarrays With Sum](https://leetcode.com/problems/binary-subarrays-with-sum/) | Medium | Count subarrays |
| [974. Subarray Sums Divisible by K](https://leetcode.com/problems/subarray-sums-divisible-by-k/) | Medium | Modular arithmetic |
| [1248. Count Number of Nice Subarrays](https://leetcode.com/problems/count-number-of-nice-subarrays/) | Medium | Transform + count |

---

## Key Takeaways

| Concept | Key Point |
|---------|----------|
| **Prefix sum definition** | `prefix[i]` = sum of elements 0 to i |
| **Range sum formula** | `sum[i,j]` = `prefix[j] - prefix[i-1]` |
| **Why prepend 0** | Simplifies edge cases for subarrays starting at index 0 |
| **Hash map purpose** | O(1) lookup of previous prefix sums |
| **Count subarrays** | Store frequency: `{prefix_sum: count}`, init `{0: 1}` |
| **Longest subarray** | Store first index: `{prefix_sum: index}`, init `{0: -1}` |
| **Order matters** | Check hash map BEFORE updating it |
| **Transform trick** | Convert problem (e.g., 0 → -1) to use prefix sum |

### The Universal Template

```python
def prefix_sum_with_hashmap(nums, target):
    # For counting: prefix_map = {0: 1}
    # For longest: prefix_map = {0: -1}
    prefix_map = {0: initial_value}
    prefix_sum = 0
    result = 0
    
    for i, num in enumerate(nums):
        prefix_sum += transform(num)  # e.g., num or (1 if num else -1)
        
        # Check if (prefix_sum - target) exists
        if (prefix_sum - target) in prefix_map:
            # For counting: result += prefix_map[prefix_sum - target]
            # For longest: result = max(result, i - prefix_map[prefix_sum - target])
            pass
        
        # Update hash map
        # For counting: prefix_map[prefix_sum] += 1
        # For longest: if prefix_sum not in prefix_map: prefix_map[prefix_sum] = i
        pass
    
    return result
```